In [ ]:
%load_ext autoreload
%autoreload 2

# DynaSurvCausalOnline Model Validation
This notebook implements the validation checks specified in `latex/model_validation.tex`.

## Objectives
1. **Factual Consistency**: Calibration and discriminative performance (C-index, Brier Score).
2. **Counterfactual Stability**: Pairwise constraints and temporal coherence.
3. **Clinical Plausibility**: Stratified analysis by biomarkers.
4. **Policy Evaluation**: Regret analysis.

In [ ]:
from warnings import warn

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from sksurv.functions import StepFunction
from sksurv.nonparametric import CensoringDistributionEstimator, kaplan_meier_estimator
from sksurv.util import Surv
from torchsurv.metrics.brier_score import BrierScore
from torchsurv.metrics.cindex import ConcordanceIndex
from torchsurv.stats.ipcw import get_ipcw
from tqdm import tqdm

from CausalSurv.data.datamodule_cv import ESMEOnlineDataModuleCV
from CausalSurv.model.DynaSurvCausalOnline import DynaSurvCausalOnline

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
# Update these paths as needed or load from a central config
MODEL_PATH = "/Users/malek/TheLAB/DynaSurv/models/HR+HER2-/4lines/06032026_141113_seed_2021779468/checkpoints/dynaSurvCausalOnline-epoch=164-val_loss= 1.1227.ckpt"
DATA_PATH = "/Users/malek/TheLAB/DynaSurv/data"
CONFIG_PATH = "/Users/malek/TheLAB/DynaSurv/configs/config_train.toml"
SPLIT_SEED = 2021779468
SUBTYPE = "HR+HER2-"
N_LINES = 4
BATCH_SIZE = 256

##  Load Model and Data

In [ ]:
# # Load Model
print(f"Loading model from {MODEL_PATH}...")
model = DynaSurvCausalOnline.load_from_checkpoint(
    MODEL_PATH, map_location=torch.device("cpu")
)
model.eval()
model.freeze()
print("Model loaded.")

# Load Data
print("Loading data...")
data_module = ESMEOnlineDataModuleCV(
    data_dir=DATA_PATH,
    subtype=SUBTYPE,
    n_intervals=100 - 1,
    n_lines=N_LINES,
    batch_size=BATCH_SIZE,
    final_training=True,
    num_workers=4,
    split_seed=SPLIT_SEED,
)
data_module.prepare_data()
data_module.setup()
print("Data loaded.")

In [ ]:
data_module.valid_treatments_per_line

In [ ]:
data_module.treatment_dict

## Treatment distribution

In [ ]:
train_dataset = data_module.cv_dataset
traetment_dics = data_module.treatment_dict
pat_idx = []
for idx in range(len(train_dataset)):
    *a , pat_id = train_dataset[idx]
    pat_idx.append(pat_id)

df = data_module.df_dynamic

# top_categories = df["T_treatment_category"].value_counts().nlargest(7).index
# df["T_treatment_category"] = df["T_treatment_category"].apply(
#     lambda x: x if x in top_categories else "OTHER"
# )

df_counts = (
    df.loc[df["lineid"] <= 4].groupby(["lineid", "T_treatment_category"]).size().reset_index(name="count")
)

pivot_df = df_counts.pivot(
    index="lineid", columns="T_treatment_category", values="count"
).fillna(0)



pivot_df

In [ ]:
pivot_df.plot(
    kind="bar",
    stacked=True,
    width=0.75,
    edgecolor="white",  # White border between segments for "clean" look
    linewidth=0.5,
)

## Factual Consistency Check

In [ ]:
test_loader = data_module.test_dataloader()
interval_bounds = model.interval_bounds.to("cpu")
trainer = L.Trainer()
model.fit_censoring_estimator(data_module.train_dataloader())
trainer.test(model, dataloaders=data_module)

## Predicted survival:

In [ ]:
def plot_predicted_survival_sample(test_dataloader, n_samples, ncols=5):
    (XPd,X_static,interval_idx,treatment_indices,time,event,mask,patient_id) = next(iter(test_dataloader))
    
    sample_idx = np.random.choice(np.arange(XPd.shape[0]), n_samples)
    
    nrows = (n_samples + ncols -1) // ncols
    fig, ax = plt.subplots(nrows, ncols, figsize=(15, 3 * nrows))
    print(ax.shape)
    surv = model.predict_discrete_survival(XPd, X_static, gather=True, factual_idx = treatment_indices)
    surv_np = surv.numpy()
    for c, pat_idx in enumerate(sample_idx):
        i = c // ncols
        j = c % ncols
        for line in range(surv_np.shape[1]):
            if not mask[pat_idx, line].bool().any():
                continue
            if nrows == 1:
                ax[j].step(x=interval_bounds,y=surv_np[pat_idx,line], label=f"line {line}")
                ax[j].set_title(f"Patient {patient_id[pat_idx]}")
                ax[j].legend()
            else:
                ax[i,j].step(x=interval_bounds,y=surv_np[pat_idx,line], label=f"line {line}")
                ax[i,j].set_title(f"Patient {patient_id[pat_idx]}")
                ax[i,j].legend()
    plt.suptitle("Sample survival predictions")
    plt.tight_layout()
    plt.show()
  
plot_predicted_survival_sample(data_module.test_dataloader(), 5)


In [ ]:
def plot_times_distribution(test_dataloader):
    (XPd, *_,time,_,mask,_) = next(iter(test_dataloader))
    N_LINES = XPd.shape[1]
    fig, ax = plt.subplots(nrows=1, ncols=N_LINES, figsize=(15, 3))
    for line in range(N_LINES):
        valid_mask = (mask[:,line] == 1)
        if not valid_mask.any():
            continue
        ax[line].hist(time[valid_mask,line], label=f"Line {line+1}", bins=50)
        ax[line].legend()        
    plt.show()
    plt.suptitle("Event time histograms")
        
plot_times_distribution(data_module.test_dataloader())

In [ ]:
def plot_km_per_line(model, test_dataloader):
    (XPd,X_static,interval_idx,treatment_indices,time,event,mask,patient_id) = next(iter(test_dataloader))
    
    fig, ax = plt.subplots(nrows=1, ncols=2, figsize = (15, 3))
    N_LINES = XPd.shape[1]
    for line in range(N_LINES):
        valid_mask = mask[:,line]==1
        e_line = model.train_events[line].ravel()
        t_line = model.train_times[line].ravel()

        surv_times, surv_probs = kaplan_meier_estimator(event=event.squeeze()[valid_mask, line].bool(), #type: ignore
                                              time_exit=time.squeeze()[valid_mask, line])
        train_times, train_prob = kaplan_meier_estimator(e_line,t_line) # type: ignore 
        
        ax[1].step(train_times, train_prob, label=f"Line {line}")
        ax[1].legend()
        ax[1].set_title("train")

        ax[0].step(surv_times, surv_probs, label=f"Line {line}")
        ax[0].legend()
        ax[0].set_title("test")

    plt.tight_layout()
    plt.suptitle("Kaplan Meier")
    plt.show()
        
plot_km_per_line(model, data_module.test_dataloader())

In [ ]:
from icecream import ic

def plot_ibs(model, test_dataloader):
    (XPd,X_static,interval_idx,treatment_indices,test_time,test_event,test_mask,patient_id) = next(iter(test_dataloader))
    horizon_times = [100, 75, 50, 30]

    assert model.train_events is not None 
    assert model.train_times is not None
    disc_surv = model.predict_discrete_survival(XPd, X_static, gather = True, factual_idx= treatment_indices)
    N_LINES = len(model.train_times.keys())
    # header = ["mean", "median", "max", "min", "95%-ile", "99%-ile"]
    # rows = []
    fig, ax = plt.subplots(nrows=1, ncols=N_LINES, figsize=(20, 3))
    for line in range(N_LINES):
        mask_line = (test_mask[:, line] == 1)
        e_line_test = test_event[mask_line, line]
        t_line_test = test_time[mask_line, line]

        e_line = model.train_events[line]
        t_line = model.train_times[line]

        # ic(e_line.shape, t_line.shape, e_line_test.shape, t_line_test.shape, disc_surv[mask_line, line].shape)

        ibs, bs_val, bs_ipcw = model.eval_brier_score_ipcw(
            train_events = torch.tensor(e_line, dtype=torch.bool), 
            train_times = torch.tensor(t_line),
            test_events = e_line_test.bool(), 
            test_times = t_line_test,
            discrete_survival = disc_surv[mask_line,line],
            tmax = horizon_times[line]
        )

        ax[line].plot(torch.linspace(0, horizon_times[line], 100),bs_val)
        ax[line].set_xticks(torch.arange(0, horizon_times[line], 6))
        
    plt.tight_layout()
    plt.show()

        

plot_ibs(model, data_module.test_dataloader())

## Calibration Plot

In [ ]:
def compute_calibration(pred_surv, obs_times, obs_events, eval_time, n_bins=10, bin_min_samples=10):
    pred_surv = pred_surv.numpy()

    bins = np.linspace(0, 1, n_bins + 1)  # discreatize the [0,1] interval into n_bins
    bin_indices = np.digitize(pred_surv, bins) - 1  #(batch, n_lines,)

    calib_pred = []
    calib_obs = []

    for i in range(n_bins):
        idx = bin_indices == i
        if idx.sum() < bin_min_samples:
            # warn(f"Bin with {idx.sum()} < bin_min_sample = {bin_min_samples}")
            continue
        # ic(bins[i], bins[i+1])
        agg_pred = np.mean(pred_surv[idx]) 
        time, survival_prob = kaplan_meier_estimator( # type:ignore
            obs_events.squeeze()[idx].astype(bool),
            obs_times.squeeze()[idx],
        )
        if eval_time <= time[0]:
            est = 1.0
            # ic("out of bound",time[0], time[-1], eval_time)
        elif eval_time >= time[-1]:
            est = survival_prob[-1]
            # ic("out of bound",time[0], time[-1], eval_time)        
        else:
            est = survival_prob[time <= eval_time][-1]


        calib_obs.append(est)
        calib_pred.append(agg_pred)

    return np.array(calib_pred), np.array(calib_obs)

def plot_calibration(eval_times, dataloader, num_bins):

    (XPd,X_static,interval_idx,treatment_indices,time,event,mask,patient_id) = next(iter(dataloader))
    disc_surv = model.predict_discrete_survival(XPd, X_static, gather=True, factual_idx = treatment_indices)
    print(disc_surv.shape)

    fig, ax = plt.subplots(nrows=1, ncols=N_LINES, figsize=(5 * N_LINES, 4))
    for line in range(N_LINES):
        valid_mask = (mask[:,line] == 1)
        if not valid_mask.any():
            continue
        pred_surv = model.eval_factual_survival(disc_surv[:, line, :], eval_times)
        # ic(line)
        # print(f"Line {line}: n_patients = {mask}")
        for i, eval_time in enumerate(eval_times):
            # ic(line)
            calib_pred, calib_obs= compute_calibration(
                pred_surv[valid_mask,i],
                time[valid_mask,line].numpy(),
                event[valid_mask,line].numpy(),
                eval_time.numpy(),
                num_bins,
            )
                
            ax[line].plot(
                calib_pred,
                calib_obs,
                ".-",
                label=f"t={eval_time}",
            )

            ax[line].set_xlabel("Predicted Survival")
        ax[line].plot([0, 1], [0, 1], "k--")
        ax[line].set_ylabel("Observed Survival (KM)")
        ax[line].legend()
        ax[line].set_title(f"Calibration Plot - Line {line + 1}")
        ax[line].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_calibration(torch.tensor([
                            #    1, 
                               2, 
                               6, 
                               12, 
                               24, 
                               30, 
                               36,
                               ]), data_module.test_dataloader(), num_bins=20)

## Adversarial unconfounding
Check if the learned representation is not predictive of the treatment

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

probabilities = {}
aacuracy = {}

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        (
            XPd,
            X_static,
            interval_idx,
            treatment_indices,
            time,
            event,
            mask,
            patient_id,
        ) = batch
        print(XPd.shape)
        h, c, p = model._init_lstm_states(X_static, XPd.device)
        for line in range(XPd.shape[1]):
            _, h, c, p = model.lstm(XPd[:, line, :], (h, c, p))

            mask_line = mask[:, line].bool()
            if not mask_line.any():
                continue

            X = h[mask_line].numpy()
            y = treatment_indices[mask_line, line].numpy()
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )

            rfc = RandomForestClassifier(n_estimators=100, random_state=42)
            rfc.fit(X_train, y_train)
            rfc.predict_proba(X_test)
            acc = rfc.score(X_test, y_test)
            aacuracy[line] = acc
            probabilities[line] = rfc.predict_proba(X_test)

print("\n--- Treatment Assignment Accuracy ---")
for line in range(N_LINES):
    acc = aacuracy.get(line, None)
    if acc is not None:
        print(f"Line {line + 1}: Treatment Assignment Accuracy = {acc:.4f}")
    else:
        print(f"Line {line + 1}: No valid data for treatment assignment accuracy.")

In [ ]:
t = model._compute_treatment_prediction_auc(XPd, X_static, treatment_indices, mask)

In [ ]:
import umap
import matplotlib.pyplot as plt
import seaborn as sns


def plot_umap_deconfounding(
    hidden_states, treatment_labels, title="UMAP Projection of LSTM Hidden States"
):
    """
    hidden_states: numpy array of shape (n_samples, hidden_dim)
    treatment_labels: 1D array of treatment categories
    """
    # 1. Initialize and fit UMAP
    reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",  # Cosine is often better for high-dim hidden states
    )
    embedding = reducer.fit_transform(hidden_states)

    # 2. Prepare DataFrame for plotting
    df = pd.DataFrame(embedding, columns=["UMAP 1", "UMAP 2"])
    df["Treatment"] = treatment_labels

    # print(df['Treatment'].value_counts())
    # 3. Create the plot
    plt.figure(figsize=(5, 5))
    sns.scatterplot(
        data=df,
        x="UMAP 1",
        y="UMAP 2",
        hue="Treatment",
        palette="viridis",
        alpha=0.6,
        s=15,
    )

    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

probabilities = {}
accuracies = {}
auc_scores = {}
dummy_auc_scores = {}

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        (
            XPd,
            X_static,
            interval_idx,
            treatment_indices,
            time,
            event,
            mask,
            patient_id,
        ) = batch
        h, c, p = model._init_lstm_states(X_static, XPd.device)

        for line in range(XPd.shape[1]):
            _, h, c, p = model.lstm(XPd[:, line, :], (h, c, p))

            mask_line = mask[:, line].bool()
            if not mask_line.any():
                continue

            X = h[mask_line].cpu().numpy()
            y = treatment_indices[mask_line, line].cpu().numpy()
            # print(np.unique(y, return_counts=True))
            # plot_umap_deconfounding(X, y, title=f"UMAP Projection - Line {line + 1}")

            unique_classes, counts = np.unique(y, return_counts=True)
            valid_classes = unique_classes[counts >= 5]
            valid_indices = np.isin(y, valid_classes)
            X = X[valid_indices]
            y = y[valid_indices]

            # print(X.shape, y.shape)
            if len(np.unique(y)) < 2:
                continue

            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=y
            )
            rfc = RandomForestClassifier(n_estimators=100, random_state=42)
            rfc.fit(X_train, y_train)
            acc = rfc.score(X_test, y_test)
            rfc_probs = rfc.predict_proba(X_test)

            dummy = DummyClassifier(strategy="stratified", random_state=42)
            dummy.fit(X_train, y_train)
            dummy_probs = dummy.predict_proba(X_test)

            # print(f"random forest probs:{rfc_probs.shape}, y_test.shape:{y_test.shape}")
            # print(f"dummy dummy_probs.shape:{dummy_probs.shape}, y_test.shape:{y_test.shape}")

            rfc_auc = roc_auc_score(
                y_test,
                rfc_probs,
                multi_class="ovr",
                average="macro",
                labels=rfc.classes_,
            )
            dummy_auc = roc_auc_score(
                y_test,
                dummy_probs,
                multi_class="ovr",
                average="macro",
                labels=dummy.classes_,
            )

            # Store results
            accuracies[line] = acc
            auc_scores[line] = rfc_auc
            dummy_auc_scores[line] = dummy_auc
            probabilities[line] = rfc_probs

# --- Print Results Table ---
print(f"\n{'Line':<10} | {'RFC Accuracy':<15} | {'RFC AUC':<10} | {'Dummy AUC':<10}")
print("-" * 55)
for line in range(N_LINES):
    if line in accuracies:
        print(
            f"Line {line + 1:<5} | {accuracies[line]:.4f}         | {auc_scores[line]:.4f}     | {dummy_auc_scores[line]:.4f}"
        )
    else:
        print(f"Line {line + 1:<5} | No valid data.")

## Treatment Policy

In [ ]:
# Regret Analysis
regrets = {line: [] for line in range(N_LINES)}

print("Computing Regret...")
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Regret"):
        (
            XPd,
            X_static,
            interval_idx,
            treatment_indices,
            time,
            event,
            mask,
            patient_id,
        ) = batch
        for line in range(XPd.shape[2]):
            mask_line = mask[:,line].bool()

            discrete_survival = model.predict_discrete_survival(XPd, X_static)
            survival = discrete_survival[mask_line]

            dt = interval_bounds[1] - interval_bounds[0]  
            utility = survival.sum(dim=3) * dt  # (batch, n_lines, n_treatments)
            ic(utility.shape)
            
            max_utility, _ = utility.max(dim=2)

            idx_expanded = treatment_indices[mask_line].unsqueeze(-1)  # (batch, n_lines, 1)
            obs_utility = torch.gather(utility, 2, idx_expanded).squeeze(2)

            regret = max_utility - obs_utility

            for line in range(N_LINES):
                valid = mask[:, line].bool()
                if valid.any():
                    regrets[line].extend(regret[valid, line].cpu().numpy())

# Histograms
plt.figure(figsize=(12, 4))
for line in range(N_LINES):
    plt.subplot(1, N_LINES, line + 1)
    plt.hist(regrets[line], bins=30, alpha=0.7)
    plt.title(f"Regret Dist - Line {line + 1}")
    plt.xlabel("Regret (RMST Difference)")
plt.tight_layout()
plt.show()

## Virtual clinical trial

In [ ]:
concordant_group = {line: {"time": [], "event": []} for line in range(N_LINES)}
discordant_group = {line: {"time": [], "event": []} for line in range(N_LINES)}

print("Running Virtual Clinical Trial (Concordance Analysis)...")

model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating Policy"):
        (
            XPd,
            X_static,
            interval_idx,
            treatment_indices,
            time,
            event,
            mask,
            patient_id,
        ) = batch

        hazards = torch.sigmoid(model(XPd, X_static)[0])
        survival_curves = torch.cumprod(1 - hazards, dim=3)

        dt = interval_bounds[1] - interval_bounds[0]
        rmst = survival_curves.sum(dim=3) * dt  # (batch, n_lines, n_treatments)

        recommended_treatment = rmst.argmax(dim=2)  # (batch, n_lines)
        observed_treatment = treatment_indices  # (batch, n_lines)

        for line in range(N_LINES):
            valid_mask = mask[:, line].bool()
            if not valid_mask.any():
                continue

            rec = recommended_treatment[valid_mask, line].cpu().numpy()
            obs = observed_treatment[valid_mask, line].cpu().numpy()

            real_t = time[valid_mask, line].cpu().numpy()
            real_e = event[valid_mask, line].cpu().numpy()

            matches = rec == obs

            concordant_group[line]["time"].extend(real_t[matches])
            concordant_group[line]["event"].extend(real_e[matches])

            discordant_group[line]["time"].extend(real_t[~matches])
            discordant_group[line]["event"].extend(real_e[~matches])

fig, axes = plt.subplots(1, N_LINES, figsize=(6 * N_LINES, 5))
if N_LINES == 1:
    axes = [axes]

for line in range(N_LINES):
    ax = axes[line]

    kmf_concordant = KaplanMeierFitter()
    kmf_discordant = KaplanMeierFitter()

    T_conc, E_conc = concordant_group[line]["time"], concordant_group[line]["event"]
    T_disc, E_disc = discordant_group[line]["time"], discordant_group[line]["event"]

    if len(T_conc) > 0:
        kmf_concordant.fit(
            T_conc, event_observed=E_conc, label=f"Matched Policy (n={len(T_conc)})"
        )
        kmf_concordant.plot_survival_function(ax=ax, ci_show=True, color="green")

    if len(T_disc) > 0:
        kmf_discordant.fit(
            T_disc, event_observed=E_disc, label=f"Differed Policy (n={len(T_disc)})"
        )
        kmf_discordant.plot_survival_function(ax=ax, ci_show=True, color="red")

    if len(T_conc) > 0 and len(T_disc) > 0:
        results = logrank_test(
            T_conc, T_disc, event_observed_A=E_conc, event_observed_B=E_disc
        )
        p_value = results.p_value

        t_max = min(np.max(T_conc), np.max(T_disc))
        rmst_conc = kmf_concordant.predict(range(int(t_max))).sum()
        rmst_disc = kmf_discordant.predict(range(int(t_max))).sum()
        delta_rmst = rmst_conc - rmst_disc

        title = (
            f"Line {line + 1}\nLog-Rank p={p_value}\nRMST Diff: {delta_rmst:.2f} months"
        )
    else:
        title = f"Line {line + 1} (Insufficient Data)"

    ax.set_title(title)
    ax.set_xlabel("Observed Time")
    ax.set_ylabel("Survival Probability")
    ax.grid(True, alpha=0.3)

fig.suptitle("Model Recommendation vs. Actual Treatment")
plt.tight_layout()
plt.show()

## Treatment recommendation analysis

# 🚧🚧🚧🚧🚧🚧🚧

In [ ]:
# from sksurv.metrics import brier_score, cumulative_dynamic_auc
# from sksurv.nonparametric import kaplan_meier_estimator
# import numpy as np
# import matplotlib.pyplot as plt

# def plot_calibration_improved(preds_surv, obs_times, obs_events, time_point,
#                                n_bins=10, strategy='quantile'):
#     """
#     Better calibration plot with more robust binning and error bars.

#     Parameters:
#     - strategy: 'uniform' (equal width) or 'quantile' (equal sample size per bin)
#     """

#     # Remove invalid predictions
#     valid_mask = ~np.isnan(preds_surv) & (preds_surv >= 0) & (preds_surv <= 1)
#     preds_surv = preds_surv[valid_mask]
#     obs_times = obs_times[valid_mask]
#     obs_events = obs_events[valid_mask]

#     if len(preds_surv) < n_bins * 5:
#         print(f"Warning: Only {len(preds_surv)} samples, reducing bins")
#         n_bins = max(3, len(preds_surv) // 10)

#     # Better binning strategy
#     if strategy == 'quantile':
#         # Equal number of samples per bin
#         bin_edges = np.percentile(preds_surv, np.linspace(0, 100, n_bins + 1))
#         bin_edges[-1] += 1e-8  # Ensure last bin includes max value
#     else:
#         # Equal width bins
#         bin_edges = np.linspace(0, 1, n_bins + 1)

#     bin_indices = np.digitize(preds_surv, bin_edges) - 1
#     bin_indices = np.clip(bin_indices, 0, n_bins - 1)  # Handle edge cases

#     calib_pred = []
#     calib_obs = []
#     calib_counts = []
#     calib_se = []  # Standard errors for confidence intervals

#     for i in range(n_bins):
#         idx = bin_indices == i
#         count = idx.sum()

#         if count < 5:  # Skip bins with too few samples
#             continue

#         # Average prediction in bin
#         avg_pred = preds_surv[idx].mean()

#         # Kaplan-Meier estimate at time_point
#         try:
#             time, survival_prob = kaplan_meier_estimator(
#                 obs_events[idx].astype(bool),
#                 obs_times[idx]
#             )

#             # Interpolate at time_point
#             if time_point <= time[0]:
#                 obs_surv = 1.0
#                 se = 0.0
#             elif time_point >= time[-1]:
#                 obs_surv = survival_prob[-1]
#                 # Greenwood's formula for KM variance (simplified)
#                 se = np.sqrt(survival_prob[-1] * (1 - survival_prob[-1]) / count)
#             else:
#                 # Find surrounding time points
#                 idx_after = np.searchsorted(time, time_point)
#                 obs_surv = survival_prob[idx_after - 1]  # Step function
#                 se = np.sqrt(obs_surv * (1 - obs_surv) / count)

#             calib_obs.append(obs_surv)
#             calib_pred.append(avg_pred)
#             calib_counts.append(count)
#             calib_se.append(se)

#         except Exception as e:
#             print(f"Skipping bin {i}: {e}")
#             continue

#     # Convert to arrays
#     calib_pred = np.array(calib_pred)
#     calib_obs = np.array(calib_obs)
#     calib_se = np.array(calib_se)

#     # Plot with error bars
#     plt.figure(figsize=(7, 7))

#     # Perfect calibration
#     plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label="Perfect Calibration", alpha=0.7)

#     # Calibration curve with confidence intervals
#     plt.errorbar(calib_pred, calib_obs, yerr=1.96*calib_se,
#                  fmt='o-', capsize=5, capthick=2,
#                  label=f"Observed (n={len(preds_surv)})")

#     # Add sample size annotations
#     for pred, obs, count in zip(calib_pred, calib_obs, calib_counts):
#         plt.annotate(f'n={count}', (pred, obs),
#                     textcoords="offset points", xytext=(0,10),
#                     ha='center', fontsize=8, alpha=0.6)

#     plt.xlabel("Predicted Survival Probability", fontsize=12)
#     plt.ylabel("Observed Survival Probability (KM)", fontsize=12)
#     plt.title(f"Calibration Plot at t={time_point:.1f}", fontsize=14)
#     plt.legend(loc='best')
#     plt.grid(True, alpha=0.3)
#     plt.xlim(-0.05, 1.05)
#     plt.ylim(-0.05, 1.05)
#     plt.tight_layout()

#     # Calculate calibration metrics
#     ece = np.sum(calib_counts * np.abs(calib_pred - calib_obs)) / sum(calib_counts)
#     print(f"Expected Calibration Error (ECE): {ece:.4f}")

#     return calib_pred, calib_obs, ece

In [ ]:
# from lifelines import KaplanMeierFitter
# from scipy.stats import kstest

# def d_calibration(preds_surv, obs_times, obs_events, time_point):
#     """
#     D-calibration: Tests if distribution of predictions matches observed.
#     Uses probability integral transform (PIT).
#     """

#     # Compute PIT values
#     pit_values = []
#     for pred, time, event in zip(preds_surv, obs_times, obs_events):
#         if time < time_point:
#             # Event occurred before time_point
#             pit = pred if event else 1.0
#         else:
#             # Censored or survived past time_point
#             pit = pred
#         pit_values.append(pit)

#     pit_values = np.array(pit_values)

#     # If well-calibrated, PIT should be uniform [0,1]
#     ks_stat, p_value = kstest(pit_values, 'uniform')

#     print(f"KS test: statistic={ks_stat:.4f}, p-value={p_value:.4f}")

#     # Plot PIT histogram
#     plt.figure(figsize=(8, 5))
#     plt.hist(pit_values, bins=20, density=True, alpha=0.7, edgecolor='black')
#     plt.axhline(1.0, color='red', linestyle='--', label='Perfect calibration (uniform)')
#     plt.xlabel('PIT Values')
#     plt.ylabel('Density')
#     plt.title(f'D-Calibration at t={time_point:.1f}')
#     plt.legend()
#     plt.tight_layout()

#     return ks_stat, p_value

In [ ]:
# # Evaluate all treatment lines and time points
# N_LINES = 3  # Adjust to your number of lines

# for line_idx in range(N_LINES):
#     print(f"\n{'='*60}")
#     print(f"Treatment Line {line_idx + 1}")
#     print(f"{'='*60}")

#     # Create subplot figure for this line
#     n_timepoints = len(model.interval_bounds) - 2
#     fig, axes = plt.subplots(1, min(3, n_timepoints), figsize=(15, 5))
#     if n_timepoints == 1:
#         axes = [axes]

#     for tp_idx in range(1, min(4, len(model.interval_bounds)-1)):
#         # Gather data for this line and time point
#         preds_surv = []
#         obs_times = []
#         obs_events = []
#         time_val = model.interval_bounds[tp_idx].item()

#         for batch_res in all_predictions:
#             mask_line = batch_res["mask"][:, line_idx].astype(bool)
#             if not mask_line.any():
#                 continue
#             haz_logits = batch_res["hazards_logit"][mask_line, line_idx, :]
#             haz = 1 / (1 + np.exp(-haz_logits))
#             surv = np.cumprod(1 - haz, axis=1)
#             preds_surv.extend(surv[:, tp_idx])
#             obs_times.extend(batch_res["time"][mask_line, line_idx])
#             obs_events.extend(batch_res["event"][mask_line, line_idx])

#         preds_surv = np.array(preds_surv)
#         obs_times = np.array(obs_times)
#         obs_events = np.array(obs_events)

#         print(f"\nTime point {tp_idx} (t={time_val:.2f}):")

#         # Run calibration
#         calib_pred, calib_obs, ece = plot_calibration_improved(
#             preds_surv, obs_times, obs_events, time_val, n_bins=10
#         )

#         # Run D-calibration
#         ks_stat, p_value = d_calibration(
#             preds_surv, obs_times, obs_events, time_val
#         )

#     plt.show()